# 🧠 Notebook 06: TISC and VM Execution

## 1. Purpose + Scope

This notebook demonstrates the core execution pipeline of T81:

*   **TISC Instruction Encoding**: How high-level logic becomes bytecode.
*   **Register Model**: The R0–R80 register file.
*   **Execution Stepping**: Observing state changes instruction-by-instruction.
*   **Disassembly**: Converting binary back to readable assembly.

## 2. Spec References

*   `spec/t81vm-spec.md`
*   `include/t81/vm/vm.hpp`
*   `include/t81/tisc/program.hpp`

## 3. Determinism Tier

**Tier A (Strict Determinism)**: The VM is a deterministic state machine. Given an initial state and a program, the sequence of states is invariant.

## 4. Reproducibility Setup

Ensure `t81_python` is built and available in `PYTHONPATH`.

In [1]:
import sys
import os

build_dir = os.path.abspath(os.path.join(os.getcwd(), "../build"))
if build_dir not in sys.path:
    sys.path.append(build_dir)

try:
    import t81_python
    print("✅ t81_python loaded.")
except ImportError:
    print("❌ Failed to load t81_python.")
    sys.exit(1)

✅ t81_python loaded.


## 5. Compilation: From Source to Bytecode

We start with a simple T81Lang program and compile it to TISC bytecode using the `compile` binding.

In [2]:
source_code = """
fn main() -> T81BigInt {
    let a: T81BigInt = 10t81;
    let b: T81BigInt = 20t81;
    return a + b;
}
"""

print("Source Code:")
print(source_code)

try:
    # Compile to bytecode (Program object)
    program = t81_python.compile(source_code)
    print("✅ Compilation successful.")
except RuntimeError as e:
    print(f"❌ Compilation failed: {e}")

Source Code:

fn main() -> T81BigInt {
    let a: T81BigInt = 10t81;
    let b: T81BigInt = 20t81;
    return a + b;
}

✅ Compilation successful.


## 6. VM Loading and Execution

Now we load the program into a fresh VM instance and execute it.

In [3]:
# Create a VM
vm = t81_python.make_interpreter_vm()

# Load program
vm.load_program(program)

# Run to completion
try:
    vm.run_to_halt(max_steps=1000)
    print("✅ Execution completed.")
except RuntimeError as e:
    print(f"❌ Runtime error: {e}")

# Inspect result register (often R0 or specialized return register depending on convention)
# For this simple function, result is likely in R0 or R1. Let's inspect.
res = vm.get_register(0)
print(f"Result in R0: {res}")

✅ Execution completed.
Result in R0: 0


[VM] push_axion_event: opcode=0 reason="meta slot axion event segment=meta addr=1293"
[VM] push_axion_event: opcode=17 reason="memory load stack addr=268 size=1"
[VM] push_axion_event: opcode=0 reason="meta slot axion event segment=meta addr=1294"
[VM] push_axion_event: opcode=16 reason="memory store stack addr=268 size=1"
[VM] push_axion_event: opcode=0 reason="meta slot axion event segment=meta addr=1295"
[VM] push_axion_event: opcode=16 reason="memory store stack addr=267 size=1"
[VM] push_axion_event: opcode=0 reason="meta slot axion event segment=meta addr=1296"
[VM] push_axion_event: opcode=17 reason="memory load stack addr=268 size=1"


## 7. Execution Tracing

We can inspect the execution trace to see the sequence of operations.

In [4]:
trace = vm.trace
print("Execution Trace (Last 10 steps):")
for entry in trace[-10:]:
    print(entry)

Execution Trace (Last 10 steps):
PC=1 OP=26
PC=4 OP=17
PC=5 OP=2
PC=6 OP=2
PC=7 OP=5
PC=8 OP=16
PC=9 OP=16
PC=10 OP=27
PC=2 OP=17
PC=3 OP=1


## 8. Failure Mode Demonstration

Infinite loops are caught by the step limit (gas/cycle limit).

In [5]:
loop_source = """
fn main() -> T81BigInt {
    var i: T81BigInt = 0t81;
    loop {
        i = i + 1t81;
    }
    return i;
}
"""

try:
    loop_prog = t81_python.compile(loop_source)
    vm_loop = t81_python.make_interpreter_vm()
    vm_loop.load_program(loop_prog)
    print("Running infinite loop with limit 100 steps...")
    vm_loop.run_to_halt(max_steps=100)
except RuntimeError as e:
    print(f"Caught expected timeout/trap: {e}")

Caught expected timeout/trap: Semantic Analyzer error


## 9. Architectural Commentary

The T81 VM is a register-based virtual machine designed for strict determinism. Unlike stack machines (JVM, WASM), it uses explicit registers to model data flow, which simplifies certain optimizations and aligns with modern hardware architectures.